In [ ]:
Sys.setenv(LANGUAGE = "en")
options(stringsAsFactors = FALSE)
suppressPackageStartupMessages(library(dplyr))
suppressPackageStartupMessages(library(data.table))
library(magrittr)
library(tibble)
library(stringr)


In [ ]:
load('project_TLS/1.for_bulk/TLS_ornot_bulk_data3.0.rdata')

In [ ]:
index = c('CCR7','MS4A1','IRF4','ADAMDEC1','IGHV3-30','AIM2','SLAMF7','GAPT','IGKC','UBD','ADAM28','MMP7','ICOS','IL7R','CCL19','VTCN1','MMP9','SEMA3C','PROM1','CFTR','STMN2')
index %in% rownames(Tpm_TLS_ornot) %>% table()

In [ ]:
choose_df = Tpm_TLS_ornot[index,]
choose_df = as.data.frame(t(choose_df))
choose_df = choose_df[match(Clinical_TLS_ornot$sample,rownames(choose_df)),]
choose_df %>% dim()
Clinical_TLS_ornot %>% dim()

In [ ]:
Clinical_TLS_ornot %<>% .[,c(1,2,3,4)] 

In [ ]:
Clinical_TLS_ornot$group = ifelse(stringr::str_detect(Clinical_TLS_ornot$group, "TLS"), 1, 0)

In [ ]:
myexpr = choose_df %>% t()
myexpr = log2(myexpr+1)
mysurv = Clinical_TLS_ornot[,c(1,5)] %>% setDF() %>% column_to_rownames('sample')

In [ ]:
myexpr = choose_df %>% t()
mysurv = Clinical_TLS_ornot[,c(1,2,3)] %>% setDF() %>% column_to_rownames('sample') %>% .[,c(2,1)] %>% set_colnames(c('months','status'))
mysurv$months = mysurv$months/30
if (all(colnames(myexpr) %in% rownames(mysurv))){
  warning("两个文件的patient ID是一致的")
} else{
  warning("两个文件的patient ID不一致")
}

In [ ]:
myexpr[1:3,1:4]
head(mysurv)

In [ ]:
library("glmnet")
library("survival")
cvfit = cv.glmnet(t(myexpr), mysurv$group, 
                  #10倍交叉验证，非必须限定条件，这篇文献有，其他文献大多没提
family='cox', alpha = 1
                  ) 


In [ ]:
library(repr)
fit <- glmnet(t(myexpr), mysurv$group, 
               family = "cox",alpha = 1) 
pdf("cvfit.pdf", width = 4,height = 4)
plot = plot(cvfit)
print(plot)
dev.off()

pdf("lambda_var.pdf", width = 4,height = 4)
plot(fit,xvar="lambda",label = TRUE)
print(plot)
dev.off()


In [ ]:
coef.min = coef(cvfit, s = "lambda.min") 
active.min = which(coef.min != 0)

In [ ]:
geneids <- rownames(myexpr)[active.min]
geneids

In [ ]:
index.min = coef.min[active.min]
index.min

In [ ]:
combine <- cbind(geneids, index.min)

In [ ]:
combine

In [ ]:
signature <- as.matrix(t(myexpr[geneids,])) %*% as.matrix(index.min) 


In [ ]:
signature %>% cbind(.,Clinical_TLS_ornot) %>% write.csv(.,file = 'roc_test.csv')

In [ ]:
# use ICGC-LIRI-JP
http://lifeome.net/database/hccdb/download/HCCDB18_mRNA_level3.zip
http://lifeome.net/database/hccdb/download/sample/HCCDB18.sample.zip
http://lifeome.net/database/hccdb/download/patient/HCCDB18.patient.zip

In [ ]:
setwd('/data/home/yu/Yu_project/project_TLS/1.for_bulk/4.for_caomeng/LIRI')

In [ ]:
urls <- c('http://lifeome.net/database/hccdb/download/HCCDB18_mRNA_level3.zip',
        'http://lifeome.net/database/hccdb/download/sample/HCCDB18.sample.zip',
        'http://lifeome.net/database/hccdb/download/patient/HCCDB18.patient.zip')

for (url in urls) {
  command <- paste0("wget -c ", url)
  system(command)
}

In [ ]:
file_path = list.files('/data/home/yu/Yu_project/project_TLS/1.for_bulk/4.for_caomeng/LIRI',full.names = T)
for (path in file_path) {
  command <- paste0("unzip -n ",path)
  system(command)
}

In [ ]:
LIRI_exp = fread('/data/home/yu/Yu_project/project_TLS/1.for_bulk/4.for_caomeng/LIRI/HCCDB18_mRNA_level3.txt')

In [ ]:
LIRI_patient = fread('/data/home/yu/Yu_project/project_TLS/1.for_bulk/4.for_caomeng/LIRI/HCCDB18.patient.txt')
LIRI_sample = fread('/data/home/yu/Yu_project/project_TLS/1.for_bulk/4.for_caomeng/LIRI/HCCDB18.sample.txt')

In [ ]:
LIRI_sample = LIRI_sample %>% t() %>% as.data.frame() %>% filter(V1 == 'HCC')

In [ ]:
dup_rows <- duplicated(LIRI_sample$V5)
LIRI_sample = LIRI_sample[!dup_rows,]

In [ ]:
LIRI_sample[1:4,1:5]

In [ ]:
LIRI_index = LIRI_sample %>% select(V4)
LIRI_index = LIRI_index[['V4']]

In [ ]:
LIRI_index[1:4]

In [ ]:
LIRI_patient %<>% t() %>% as.data.frame()
colnames(LIRI_patient) = LIRI_patient[1,];LIRI_patient = LIRI_patient[-1,]

In [ ]:
LIRI_patient[1:4,]

In [ ]:
LIRI_outcome = LIRI_patient[LIRI_index,]
LIRI_outcome_surv = LIRI_outcome %>% select(SUR,STATUS)
LIRI_outcome_surv$STATUS = ifelse()

In [ ]:
LIRI_outcome_surv$STATUS <- ifelse(stringr::str_detect(LIRI_outcome_surv$STATUS, "Alive"), 0, 1)
LIRI_outcome_surv$SUR = as.numeric(LIRI_outcome_surv$SUR)

In [ ]:
LIRI_exp %<>% column_to_rownames('Symbol')


In [ ]:
LIRI_exp = LIRI_exp[,-1]

In [ ]:
LIRI_exp = LIRI_exp[,rownames(LIRI_sample)]

In [ ]:
LIRI_exp[1:4,1:4]
LIRI_outcome_surv[1:4,1:2]

In [ ]:
index_df = LIRI_sample[,3:4];index_df$s_id = rownames(index_df)
index_df = index_df[,-1]
index_df[1:4,1:2]

In [ ]:
colnames(LIRI_exp) %>% length()
rownames(LIRI_outcome_surv) %>% length()

In [ ]:
signature_LIRI <- as.matrix(t(LIRI_exp[geneids,])) %*% as.matrix(index.min) 

In [ ]:
for_xt = signature_LIRI %>% as.data.frame() %>% cbind(.,LIRI_outcome_surv)
for_xt$V1 = -1*for_xt$V1
write.csv(for_xt,'project_TLS/1.for_bulk/roc_LIRI_test2.csv')

In [ ]:
save.image(file = 'project_TLS/1.for_bulk/3.lasso_cox.rdata')